# 난임 환자 임신 성공 여부 예측

이 노트북은 프로젝트 폴더 안의 `train.csv`, `test.csv`, `sample_submission.csv`를 사용하여  
EDA → feature 처리 → 모델 선택 → 5-fold 검증 → 앙상블 제출 파일 생성 순서로 재현할 수 있도록 구성했습니다.

특정 순위나 외부 결과를 전제로 하지 않고, 입력 데이터에서부터 전체 과정을 다시 실행하는 방식입니다.


## 진행 순서

1. VS Code에서 프로젝트 폴더 열기  
2. 데이터 구조 확인  
3. EDA 기반 feature 처리  
4. 모델 선택  
5. 5-fold Stratified K-Fold 검증  
6. 후보 모델 학습  
7. OOF ROC-AUC 기준 앙상블  
8. 제출 파일 생성

처음 실행할 때는 `RUN_MODE = "quick"`으로 코드가 잘 도는지 확인하고,  
제출 후보를 만들 때는 `RUN_MODE = "full"`로 바꿔 실행하세요.


### 실행 모드 해석

- `quick`: 코드가 정상 작동하는지 확인하는 모드입니다. 학습 반복 수와 모델 수가 줄어 있어 빠르지만, 제출용 성능 판단에는 적합하지 않습니다.
- `full`: 제출 후보를 만들기 위한 모드입니다. 모델 수와 반복 수가 늘어나므로 시간이 오래 걸리지만, 리더보드 제출에는 이 모드를 기준으로 보는 것이 적절합니다.
- `USE_CATBOOST_GPU=False`: VS Code 로컬 환경에서는 CPU 실행을 기본값으로 둡니다. GPU 환경이 준비되어 있을 때만 `True`로 바꾸면 됩니다.

처음에는 `quick`으로 전체 흐름을 확인하고, 제출 파일을 만들 때 `full`로 변경하는 방식이 안전합니다.


In [3]:
# VS Code 로컬 환경 준비

import importlib.util

required_packages = ["catboost", "lightgbm", "xgboost", "scipy", "tabulate"]
missing_packages = [
    pkg for pkg in required_packages
    if importlib.util.find_spec(pkg) is None
]

if missing_packages:
    print("설치가 필요한 패키지:", missing_packages)
    print("VS Code 터미널에서 아래 명령을 실행하세요.")
    print("pip install " + " ".join(missing_packages))
else:
    print("필수 패키지 확인 완료")


필수 패키지 확인 완료


In [4]:
from __future__ import annotations

import os
import re
import json
import zipfile
import itertools
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


warnings.filterwarnings("ignore")

try:
    from catboost import CatBoostClassifier, Pool
except Exception as e:
    CatBoostClassifier = None
    Pool = None
    print("CatBoost import error:", e)

try:
    import lightgbm as lgb
except Exception as e:
    lgb = None
    print("LightGBM import error:", e)

try:
    from xgboost import XGBClassifier
except Exception as e:
    XGBClassifier = None
    print("XGBoost import error:", e)


In [ ]:
# 실행 설정

RUN_MODE = "full"   # quick: 동작 확인용 / full: 제출 후보 생성용
N_SPLITS = 5
SEEDS = [42] if RUN_MODE == "quick" else [42, 2026, 777]

USE_CATBOOST_GPU = False  # VS Code 로컬 기본값은 CPU. GPU 환경이면 True로 변경

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR
OUT_DIR = PROJECT_DIR / "outputs" / "project_fertility_psp_v4"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "임신 성공 여부"
ID_COL = "ID"
MISSING_TOKEN = "__MISSING__"

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print("RUN_MODE:", RUN_MODE)
print("N_SPLITS:", N_SPLITS)
print("SEEDS:", SEEDS)


PROJECT_DIR: c:\Users\jkl83\OneDrive\Desktop\난임 환자 대상 임신 성공 여부 예측 AI 프로젝트
DATA_DIR: c:\Users\jkl83\OneDrive\Desktop\난임 환자 대상 임신 성공 여부 예측 AI 프로젝트
OUT_DIR: c:\Users\jkl83\OneDrive\Desktop\난임 환자 대상 임신 성공 여부 예측 AI 프로젝트\outputs\project_fertility_psp_v4
RUN_MODE: quick
N_SPLITS: 5
SEEDS: [42]


### 데이터 누수 방지 기준

이 노트북에서 중요한 검증 기준은 **제출용 데이터가 학습 기준 산출에 사용되지 않도록 하는 것**입니다.

- target인 `임신 성공 여부`는 train에만 존재하며, 모델 학습과 OOF 검증에만 사용합니다.
- Feature Engineering은 각 행의 값으로 만들 수 있는 규칙 기반 변수만 생성합니다.
- LightGBM/XGBoost용 인코딩은 각 fold의 학습 데이터에서만 fit하고, 해당 fold의 validation과 제출용 데이터에는 transform만 적용합니다.
- OOF 앙상블 가중치는 test 점수가 아니라 train 내부의 OOF ROC-AUC를 기준으로 선택합니다.

따라서 이 노트북의 성능 비교는 단일 제출 점수에 맞춘 사후 조정이 아니라, 학습 데이터 내부 검증 결과에 기반합니다.


In [6]:
# 로컬 데이터 파일 확인

required_files = ["train.csv", "test.csv", "sample_submission.csv"]
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        "프로젝트 폴더에 필요한 파일이 없습니다: "
        + ", ".join(missing_files)
        + f"\n현재 DATA_DIR: {DATA_DIR}"
    )

print("데이터 파일 확인 완료")
for name in required_files:
    print(DATA_DIR / name)


데이터 파일 확인 완료
c:\Users\jkl83\OneDrive\Desktop\난임 환자 대상 임신 성공 여부 예측 AI 프로젝트\train.csv
c:\Users\jkl83\OneDrive\Desktop\난임 환자 대상 임신 성공 여부 예측 AI 프로젝트\test.csv
c:\Users\jkl83\OneDrive\Desktop\난임 환자 대상 임신 성공 여부 예측 AI 프로젝트\sample_submission.csv


In [7]:
# 데이터 로드 및 기본 확인

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train:", train.shape)
print("test:", test.shape)
print("sample_submission:", sample_submission.shape)
print("target mean:", train[TARGET].mean())
print("target counts:")
display(train[TARGET].value_counts().to_frame("count"))

train: (256351, 69)
test: (90067, 68)
sample_submission: (90067, 2)
target mean: 0.2583489044318142
target counts:


,count
임신 성공 여부,
0,190123
1,66228


### 데이터 로드 결과 해석

이 셀에서는 train, test, sample submission의 기본 구조를 확인합니다.

- train에는 target 컬럼 `임신 성공 여부`가 포함됩니다.
- test는 제출용 데이터이므로 target이 없습니다.
- sample submission은 최종 제출 파일의 컬럼명과 행 순서를 맞추기 위한 기준 파일입니다.

여기서 target 평균은 전체 임신 성공률이며, 이후 모델은 각 환자/시술 행의 성공 확률을 예측합니다.


## 1. EDA에서 feature 처리로 연결한 내용

이번 데이터에서 전처리의 핵심은 단순 결측 대체가 아니라,  
**시술 구조와 결측 패턴을 모델이 구분할 수 있도록 만드는 것**입니다.

반영한 방향은 다음과 같습니다.

- DI와 IVF 시술 구조 구분
- 배아 관련 변수의 반복 결측을 구조적 결측으로 처리
- `특정 시술 유형` 문자열에서 BLASTOCYST, ICSI, AH, IUI 등의 요소 분리
- 나이 구간을 순서형 값으로 변환
- 배아 이식 경과일, 이식 배아 수, 생성 배아 수의 조합 및 비율 생성
- 나이 × 난자 출처, 이식일 × 이식 배아 수 같은 조합 변수 생성


In [8]:
# EDA 요약 테이블

def overview_tables(train):
    print("성공 비율:", round(train[TARGET].mean() * 100, 2), "%")
    if "시술 유형" in train.columns:
        display(
            train.groupby("시술 유형")[TARGET]
            .agg(["count", "mean"])
            .rename(columns={"mean": "success_rate"})
            .sort_values("count", ascending=False)
        )
    if "시술 당시 나이" in train.columns:
        display(
            train.groupby("시술 당시 나이")[TARGET]
            .agg(["count", "mean"])
            .rename(columns={"mean": "success_rate"})
            .sort_values("success_rate", ascending=False)
        )
    missing = (
        train.drop(columns=[TARGET], errors="ignore")
        .isna()
        .mean()
        .sort_values(ascending=False)
        .head(15)
        .to_frame("missing_rate")
    )
    display(missing)

overview_tables(train)

성공 비율: 25.83 %


,count,success_rate
시술 유형,,
IVF,250060,0.261605
DI,6291,0.128914


,count,success_rate
시술 당시 나이,,
만18-34세,102476,0.322622
만35-37세,57780,0.278401
만38-39세,39247,0.217138
만45-50세,6918,0.167679
만40-42세,37348,0.159393
만43-44세,12253,0.118012
알 수 없음,329,0.000000


,missing_rate
난자 해동 경과일,0.994398
PGS 시술 여부,0.992475
PGD 시술 여부,0.991500
착상 전 유전 검사 사용 여부,0.989397
임신 시도 또는 마지막 임신 경과 연수,0.963449
배아 해동 경과일,0.842525
난자 채취 경과일,0.224255
난자 혼합 경과일,0.209615
배아 이식 경과일,0.169947
저장된 배아 수,0.024541


### EDA 요약 결과 해석

이 EDA 셀은 전체 탐색을 길게 보여주기보다, 모델링에 직접 연결되는 세 가지를 빠르게 확인합니다.

- `시술 유형`별 성공률: IVF와 DI의 성공률 차이를 확인하여 시술 구조 차이가 중요한 feature임을 확인합니다.
- `시술 당시 나이`별 성공률: 나이 구간에 따라 성공률이 달라지는지 확인합니다.
- 결측률 상위 변수: 결측이 단순 오류인지, 특정 시술 프로세스가 없어서 생긴 구조적 결측인지 판단하는 출발점입니다.

이 결과를 바탕으로 이후 feature 처리에서는 결측을 무조건 삭제하거나 평균으로 채우기보다, 결측 여부와 시술 구조를 함께 반영합니다.


In [9]:
# Feature 처리 함수

EMBRYO_COLS = [
    "총 생성 배아 수",
    "미세주입된 난자 수",
    "미세주입에서 생성된 배아 수",
    "이식된 배아 수",
    "미세주입 배아 이식 수",
    "저장된 배아 수",
    "미세주입 후 저장된 배아 수",
    "해동된 배아 수",
    "해동 난자 수",
    "수집된 신선 난자 수",
    "저장된 신선 난자 수",
    "혼합된 난자 수",
    "파트너 정자와 혼합된 난자 수",
    "기증자 정자와 혼합된 난자 수",
]

COUNT_COLS = [
    "총 시술 횟수",
    "클리닉 내 총 시술 횟수",
    "IVF 시술 횟수",
    "DI 시술 횟수",
    "총 임신 횟수",
    "IVF 임신 횟수",
    "DI 임신 횟수",
    "총 출산 횟수",
    "IVF 출산 횟수",
    "DI 출산 횟수",
]

AGE_ORD = {
    "만18-34세": 0,
    "만35-37세": 1,
    "만38-39세": 2,
    "만40-42세": 3,
    "만43-44세": 4,
    "만45-50세": 5,
    "알 수 없음": np.nan,
    MISSING_TOKEN: np.nan,
}

AGE_MID = {
    "만18-34세": 29.0,
    "만35-37세": 36.0,
    "만38-39세": 38.5,
    "만40-42세": 41.0,
    "만43-44세": 43.5,
    "만45-50세": 47.5,
    "알 수 없음": np.nan,
    MISSING_TOKEN: np.nan,
}


def safe_str(df: pd.DataFrame, col: str) -> pd.Series:
    if col not in df.columns:
        return pd.Series(MISSING_TOKEN, index=df.index, dtype="string")
    return df[col].astype("string").fillna(MISSING_TOKEN)


def safe_num(df: pd.DataFrame, col: str) -> pd.Series:
    if col not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype="float64")
    return pd.to_numeric(df[col], errors="coerce")


def contains(series: pd.Series, token: str) -> pd.Series:
    return series.astype("string").fillna(MISSING_TOKEN).str.contains(
        token, case=False, regex=False, na=False
    )


def ratio(a: pd.Series, b: pd.Series) -> pd.Series:
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce").replace(0, np.nan)
    return (a / b).replace([np.inf, -np.inf], np.nan)


def count_to_ord(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce")
    extracted = s.astype("string").str.extract(r"(\d+)")[0]
    return pd.to_numeric(extracted, errors="coerce")


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    treatment = safe_str(df, "시술 유형")
    procedure = safe_str(df, "특정 시술 유형")
    age = safe_str(df, "시술 당시 나이")
    egg_source = safe_str(df, "난자 출처")
    sperm_source = safe_str(df, "정자 출처")
    reason = safe_str(df, "배아 생성 주요 이유")

    # 시술 구조
    df["f_is_ivf"] = (treatment == "IVF").astype("int8")
    df["f_is_di"] = (treatment == "DI").astype("int8")

    # 특정 시술 유형 token 분리
    df["f_has_blastocyst"] = contains(procedure, "BLASTOCYST").astype("int8")
    df["f_has_ah"] = contains(procedure, "AH").astype("int8")
    df["f_has_icsi"] = contains(procedure, "ICSI").astype("int8")
    df["f_has_ivf_token"] = contains(procedure, "IVF").astype("int8")
    df["f_has_unknown_proc"] = contains(procedure, "Unknown").astype("int8")
    df["f_is_iui"] = (procedure == "IUI").astype("int8")
    df["f_is_ici"] = (procedure == "ICI").astype("int8")
    df["f_is_ivi"] = (procedure == "IVI").astype("int8")
    df["f_proc_has_colon"] = procedure.str.contains(":", regex=False, na=False).astype("int8")
    df["f_proc_has_slash"] = procedure.str.contains("/", regex=False, na=False).astype("int8")
    df["f_proc_complex"] = ((df["f_proc_has_colon"] == 1) | (df["f_proc_has_slash"] == 1)).astype("int8")
    df["f_proc_token_count"] = (
        procedure.str.replace(" ", "", regex=False)
        .str.split(r"[:/]+", regex=True)
        .str.len()
        .fillna(0)
        .astype("int16")
    )

    # 나이
    df["f_age_ord"] = age.map(AGE_ORD).astype("float64")
    df["f_age_mid"] = age.map(AGE_MID).astype("float64")
    df["f_age_unknown"] = age.isin(["알 수 없음", MISSING_TOKEN]).astype("int8")
    df["f_age_38plus"] = (df["f_age_ord"] >= 2).astype("int8")
    df["f_age_40plus"] = (df["f_age_ord"] >= 3).astype("int8")
    df["f_age_43plus"] = (df["f_age_ord"] >= 4).astype("int8")

    # 시술/임신/출산 횟수 문자열을 순서형으로 변환
    for c in COUNT_COLS:
        if c in df.columns:
            df[f"f_ord__{c}"] = count_to_ord(df[c])

    # 구조적 결측
    emb_cols = [c for c in EMBRYO_COLS if c in df.columns]
    if emb_cols:
        df["f_emb_missing_count"] = df[emb_cols].isna().sum(axis=1).astype("int16")
        df["f_emb_all_missing"] = df[emb_cols].isna().all(axis=1).astype("int8")
        df["f_emb_any_missing"] = df[emb_cols].isna().any(axis=1).astype("int8")
        df["f_emb_missing_rate"] = df["f_emb_missing_count"] / len(emb_cols)

    # 배아 이식/생성 관련
    transfer_day = safe_num(df, "배아 이식 경과일")
    transfer_count = safe_num(df, "이식된 배아 수")
    created = safe_num(df, "총 생성 배아 수")
    stored = safe_num(df, "저장된 배아 수")
    thawed = safe_num(df, "해동된 배아 수")
    fresh_oocyte = safe_num(df, "수집된 신선 난자 수")
    mixed_oocyte = safe_num(df, "혼합된 난자 수")
    partner_mixed = safe_num(df, "파트너 정자와 혼합된 난자 수")
    donor_mixed = safe_num(df, "기증자 정자와 혼합된 난자 수")
    single_embryo = safe_num(df, "단일 배아 이식 여부")

    df["f_transfer_day_missing"] = transfer_day.isna().astype("int8")
    df["f_transfer_day_filled"] = transfer_day.fillna(-1)
    df["f_transfer_day_0_1"] = transfer_day.isin([0, 1]).astype("int8")
    df["f_transfer_day_2_3"] = transfer_day.isin([2, 3]).astype("int8")
    df["f_transfer_day_4_5"] = transfer_day.isin([4, 5]).astype("int8")
    df["f_transfer_day_5"] = (transfer_day == 5).astype("int8")
    df["f_transfer_day_6plus"] = (transfer_day >= 6).astype("int8")

    df["f_transfer_count_missing"] = transfer_count.isna().astype("int8")
    df["f_transfer_count_zero"] = (transfer_count.fillna(0) == 0).astype("int8")
    df["f_transfer_count_one"] = (transfer_count == 1).astype("int8")
    df["f_transfer_count_ge2"] = (transfer_count >= 2).astype("int8")
    df["f_no_transfer"] = ((transfer_count.fillna(0) == 0) | transfer_day.isna()).astype("int8")

    df["f_single_embryo_missing"] = single_embryo.isna().astype("int8")
    df["f_single_embryo_yes"] = (single_embryo == 1).astype("int8")

    # 상호작용
    df["f_day_x_transfer_count"] = transfer_day.fillna(-1) * transfer_count.fillna(0)
    df["f_created_x_day"] = created.fillna(0) * transfer_day.fillna(-1)
    df["f_age_x_day"] = df["f_age_ord"].fillna(-1) * transfer_day.fillna(-1)
    df["f_age_x_transfer_count"] = df["f_age_ord"].fillna(-1) * transfer_count.fillna(0)
    df["f_blastocyst_x_day5"] = df["f_has_blastocyst"] * df["f_transfer_day_5"]
    df["f_ah_x_age40"] = df["f_has_ah"] * df["f_age_40plus"]

    # 비율
    df["f_transfer_per_created"] = ratio(transfer_count, created)
    df["f_stored_per_created"] = ratio(stored, created)
    df["f_thawed_per_stored"] = ratio(thawed, stored)
    df["f_mixed_per_fresh_oocyte"] = ratio(mixed_oocyte, fresh_oocyte)
    df["f_partner_mixed_ratio"] = ratio(partner_mixed, mixed_oocyte)
    df["f_donor_mixed_ratio"] = ratio(donor_mixed, mixed_oocyte)
    df["f_created_minus_transferred"] = created.fillna(0) - transfer_count.fillna(0)
    df["f_created_minus_stored"] = created.fillna(0) - stored.fillna(0)

    # 배아 생성 목적
    has_current = contains(reason, "현재 시술용")
    has_storage = contains(reason, "배아 저장용")
    has_egg_storage = contains(reason, "난자 저장용")
    has_donation = contains(reason, "기증용")

    df["f_reason_current"] = has_current.astype("int8")
    df["f_reason_storage"] = has_storage.astype("int8")
    df["f_reason_egg_storage"] = has_egg_storage.astype("int8")
    df["f_reason_donation"] = has_donation.astype("int8")
    df["f_reason_storage_only"] = (has_storage & ~has_current & ~has_donation).astype("int8")
    df["f_reason_current_and_donation"] = (has_current & has_donation).astype("int8")

    # 유전검사 관련 고결측 변수는 값 자체보다 known/missing flag 위주로 사용
    for c in ["PGD 시술 여부", "PGS 시술 여부", "착상 전 유전 검사 사용 여부", "착상 전 유전 진단 사용 여부"]:
        if c in df.columns:
            df[f"f_known__{c}"] = (~df[c].isna()).astype("int8")
            df[f"f_missing__{c}"] = df[c].isna().astype("int8")

    # 난자/정자 출처
    df["f_oocyte_donor"] = (~egg_source.isin(["본인 제공", "알 수 없음", MISSING_TOKEN, "Unknown"])).astype("int8")
    df["f_sperm_donor"] = (~sperm_source.isin(["배우자 제공", "알 수 없음", MISSING_TOKEN, "Unknown"])).astype("int8")
    df["f_any_donor_gamete"] = ((df["f_oocyte_donor"] == 1) | (df["f_sperm_donor"] == 1)).astype("int8")
    df["f_age_x_oocyte_donor"] = df["f_age_ord"].fillna(-1) * df["f_oocyte_donor"]

    # 신선/동결 배아 사용
    frozen = safe_num(df, "동결 배아 사용 여부")
    fresh = safe_num(df, "신선 배아 사용 여부")
    donor_emb = safe_num(df, "기증 배아 사용 여부")
    df["f_frozen_used"] = (frozen == 1).astype("int8")
    df["f_fresh_used"] = (fresh == 1).astype("int8")
    df["f_donor_embryo_used"] = (donor_emb == 1).astype("int8")
    df["f_fresh_and_frozen"] = ((fresh == 1) & (frozen == 1)).astype("int8")
    df["f_neither_fresh_nor_frozen"] = ((fresh.fillna(0) == 0) & (frozen.fillna(0) == 0)).astype("int8")

    # 주요 조합 변수
    df["f_combo_day_count"] = transfer_day.fillna(-1).astype("int16").astype(str) + "__" + transfer_count.fillna(-1).astype("int16").astype(str)
    df["f_combo_age_oocyte"] = age.astype(str) + "__" + egg_source.astype(str)
    df["f_combo_age_proc"] = age.astype(str) + "__" + procedure.astype(str)
    df["f_combo_reason_proc"] = reason.astype(str) + "__" + procedure.astype(str)
    df["f_combo_day_reason"] = transfer_day.fillna(-1).astype("int16").astype(str) + "__" + reason.astype(str)
    df["f_combo_fresh_frozen"] = fresh.fillna(-1).astype("int16").astype(str) + "__" + frozen.fillna(-1).astype("int16").astype(str)

    return df


def prepare_features(train: pd.DataFrame, test: pd.DataFrame):
    y = train[TARGET].astype(int).to_numpy()

    train_x = train.drop(columns=[TARGET]).copy()
    test_x = test.copy()

    # 같은 row-wise feature 규칙을 train과 제출용 데이터에 각각 적용합니다.
    train_x = add_features(train_x)
    test_x = add_features(test_x)

    X = train_x.drop(columns=[ID_COL], errors="ignore").copy()
    X_test = test_x.drop(columns=[ID_COL], errors="ignore").copy()
    X_test = X_test.reindex(columns=X.columns)

    for df in [X, X_test]:
        for c in df.select_dtypes(include=["bool"]).columns:
            df[c] = df[c].astype("int8")

    cat_cols = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    num_cols = [c for c in X.columns if c not in cat_cols]

    for c in cat_cols:
        X[c] = X[c].astype("string").fillna(MISSING_TOKEN).astype(str)
        X_test[c] = X_test[c].astype("string").fillna(MISSING_TOKEN).astype(str)

    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")
        X_test[c] = pd.to_numeric(X_test[c], errors="coerce")

    return X, X_test, y, cat_cols, num_cols


X, X_test, y, cat_cols, num_cols = prepare_features(train, test)

print("X:", X.shape)
print("X_test:", X_test.shape)
print("categorical cols:", len(cat_cols))
print("numeric cols:", len(num_cols))

X: (256351, 158)
X_test: (90067, 158)
categorical cols: 26
numeric cols: 132


### Feature Engineering 해석

이 함수는 v4 노트북의 핵심입니다. 주요 처리 방향은 다음과 같습니다.

1. **시술 구조 변수**
- IVF/DI 여부, ICSI/IVF/BLASTOCYST/AH/IUI 등 특정 시술 유형의 문자열 token을 분리합니다.
- 단일 문자열 범주를 그대로 두는 것보다, 시술 요소를 여러 flag로 나누면 모델이 세부 시술 의미를 더 쉽게 학습할 수 있습니다.

2. **나이 변수**
- `시술 당시 나이`를 순서형 값과 대표 나이값으로 변환합니다.
- 38세 이상, 40세 이상, 43세 이상 flag를 만들어 성공률 하락 구간을 반영합니다.

3. **구조적 결측 변수**
- 배아/난자 관련 변수의 결측 개수, 전체 결측 여부, 결측률을 feature로 만듭니다.
- 특히 DI는 IVF와 달리 배아 생성·이식 과정이 없을 수 있으므로, 결측 자체가 중요한 시술 프로세스 신호가 됩니다.

4. **배아 이식 및 생성 변수**
- 배아 이식 경과일, 이식 배아 수, 총 생성 배아 수, 저장/해동 배아 수 등을 이용해 이식 단계와 배아 확보 정도를 표현합니다.
- `배아 이식 경과일 × 이식된 배아 수`, `나이 × 이식 배아 수` 같은 상호작용 변수도 함께 생성합니다.

5. **비율 변수**
- `이식 배아 수 / 총 생성 배아 수`, `저장 배아 수 / 총 생성 배아 수`처럼 단순 개수보다 효율을 나타내는 변수를 만듭니다.
- 난자·배아 수가 많다고 항상 좋은 것이 아니므로, 비율과 잔여량을 함께 보는 전략입니다.

6. **조합 변수**
- 나이 × 난자 출처, 나이 × 시술 유형, 배아 생성 이유 × 시술 유형 등의 조합을 생성합니다.
- 의료 데이터에서는 단일 변수보다 변수 조합에서 의미가 커지는 경우가 많기 때문입니다.


## 2. 모델 선택 기준

이 데이터는 범주형 변수가 많고, 결측 패턴과 변수 조합이 중요합니다.  
따라서 하나의 모델만 사용하는 대신 성격이 다른 tree boosting 모델을 함께 사용합니다.

- **CatBoost**: 범주형 변수를 직접 처리할 수 있어 기본 후보로 사용
- **LightGBM**: 빠른 학습과 다른 분할 방식으로 다양성 확보
- **XGBoost**: LightGBM과 다른 regularization/분할 특성으로 앙상블 다양성 확보

LightGBM과 XGBoost는 문자열 범주형을 그대로 넣지 않고,  
각 fold의 학습 데이터에서만 **ordinal + frequency encoding**을 fit해서 검증 데이터 누수를 막습니다.


### 모델별 역할 해석

이 노트북은 CatBoost, LightGBM, XGBoost를 함께 사용합니다. 세 모델 모두 tree boosting 계열이지만 강점이 조금씩 다릅니다.

- CatBoost는 범주형 변수를 직접 다루는 데 강하고, 이 데이터처럼 범주형/결측 패턴이 많은 경우 기본 성능이 안정적입니다.
- LightGBM은 학습 속도가 빠르고 다른 방식으로 분기 구조를 만들기 때문에 CatBoost와 다른 오류 패턴을 만들 수 있습니다.
- XGBoost는 regularization이 강한 편이라 과적합을 줄이는 후보 모델로 활용할 수 있습니다.

앙상블의 목적은 단순히 모델 수를 늘리는 것이 아니라, **서로 다른 오류 패턴을 가진 예측값을 조합해 순위 품질을 높이는 것**입니다.


In [10]:
# Fold별 범주형 인코더
# LightGBM/XGBoost용: categorical column 하나를 ord/freq 두 개로 변환

@dataclass
class FoldEncoder:
    cat_cols: List[str]
    num_cols: List[str]
    maps_: Dict[str, Dict[str, int]] | None = None
    freqs_: Dict[str, Dict[str, float]] | None = None
    medians_: Dict[str, float] | None = None

    def fit(self, X: pd.DataFrame):
        self.maps_ = {}
        self.freqs_ = {}
        self.medians_ = {}
        n = max(len(X), 1)

        for c in self.cat_cols:
            s = X[c].astype("string").fillna(MISSING_TOKEN).astype(str)
            vc = s.value_counts(dropna=False)
            self.maps_[c] = {v: i + 1 for i, v in enumerate(vc.index)}
            self.freqs_[c] = (vc / n).to_dict()

        for c in self.num_cols:
            med = pd.to_numeric(X[c], errors="coerce").median()
            self.medians_[c] = 0.0 if pd.isna(med) else float(med)

        return self

    def transform(self, X: pd.DataFrame):
        parts = []

        if self.num_cols:
            num = X[self.num_cols].copy()
            for c in self.num_cols:
                num[c] = pd.to_numeric(num[c], errors="coerce").fillna(self.medians_.get(c, 0.0))
            parts.append(num.astype("float32"))

        cat_parts = []
        for c in self.cat_cols:
            s = X[c].astype("string").fillna(MISSING_TOKEN).astype(str)
            cat_parts.append(pd.DataFrame({
                f"{c}__ord": s.map(self.maps_[c]).fillna(0).astype("int32"),
                f"{c}__freq": s.map(self.freqs_[c]).fillna(0.0).astype("float32"),
            }, index=X.index))

        if cat_parts:
            parts.append(pd.concat(cat_parts, axis=1))

        return pd.concat(parts, axis=1)

### FoldEncoder 해석

LightGBM과 XGBoost는 문자열 범주형 변수를 그대로 처리하기 어렵기 때문에, fold 내부에서 직접 인코딩합니다.

- `ord`: 각 범주를 정수 번호로 변환합니다.
- `freq`: 해당 범주가 fold 학습 데이터에서 얼마나 자주 등장했는지를 비율로 변환합니다.
- `median`: 수치형 결측값을 fold 학습 데이터의 중앙값으로 대체합니다.

중요한 점은 이 인코더가 **각 fold의 train 부분에서만 fit**된다는 것입니다.  
validation과 제출용 데이터는 이미 fit된 기준으로 transform만 하므로, validation target 정보가 인코딩에 섞이는 누수를 막을 수 있습니다.


In [11]:
# 모델 파라미터

def clip_pred(pred, eps=1e-7):
    return np.clip(np.asarray(pred, dtype=float), eps, 1.0 - eps)


def cat_params(seed, variant="a"):
    if RUN_MODE == "quick":
        iters = 500
        lr = 0.06
        od_wait = 70
    else:
        iters = 2600 if variant == "a" else 2200
        lr = 0.028 if variant == "a" else 0.032
        od_wait = 180

    params = dict(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=iters,
        learning_rate=lr,
        depth=6 if variant == "a" else 7,
        l2_leaf_reg=10.0 if variant == "a" else 14.0,
        random_strength=0.6 if variant == "a" else 0.8,
        bagging_temperature=0.3 if variant == "a" else 0.6,
        od_type="Iter",
        od_wait=od_wait,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )
    if USE_CATBOOST_GPU:
        params["task_type"] = "GPU"
    return params


def lgb_params(seed, variant="a"):
    if RUN_MODE == "quick":
        n_estimators = 600
        lr = 0.04
        early = 100
    else:
        n_estimators = 2400 if variant == "a" else 2200
        lr = 0.020 if variant == "a" else 0.024
        early = 220

    return dict(
        objective="binary",
        metric="auc",
        n_estimators=n_estimators,
        learning_rate=lr,
        num_leaves=64 if variant == "a" else 40,
        max_depth=-1 if variant == "a" else 8,
        min_child_samples=80 if variant == "a" else 100,
        subsample=0.82 if variant == "a" else 0.88,
        colsample_bytree=0.82 if variant == "a" else 0.78,
        reg_alpha=0.15 if variant == "a" else 0.25,
        reg_lambda=8.0 if variant == "a" else 12.0,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    ), early


def xgb_params(seed, variant="a"):
    if RUN_MODE == "quick":
        n_estimators = 600
        lr = 0.04
    else:
        n_estimators = 2200 if variant == "a" else 2000
        lr = 0.020 if variant == "a" else 0.024

    return dict(
        objective="binary:logistic",
        eval_metric="auc",
        n_estimators=n_estimators,
        learning_rate=lr,
        max_depth=4 if variant == "a" else 5,
        min_child_weight=14 if variant == "a" else 18,
        subsample=0.84 if variant == "a" else 0.80,
        colsample_bytree=0.84 if variant == "a" else 0.78,
        reg_alpha=0.12 if variant == "a" else 0.25,
        reg_lambda=8.0 if variant == "a" else 12.0,
        random_state=seed,
        tree_method="hist",
        n_jobs=-1,
    )

### 모델 파라미터 해석

이 셀은 quick/full 모드에 따라 학습 강도를 다르게 설정합니다.

- `iterations`, `n_estimators`: 학습 반복 횟수입니다. full 모드에서는 더 많이 학습합니다.
- `learning_rate`: 한 번에 업데이트하는 크기입니다. 작을수록 천천히 학습하지만 안정적입니다.
- `depth`, `num_leaves`, `max_depth`: 트리 복잡도를 조절합니다.
- `reg_alpha`, `reg_lambda`, `l2_leaf_reg`: 과적합을 줄이기 위한 정규화 설정입니다.
- `early stopping`: validation 성능이 더 이상 좋아지지 않으면 학습을 멈춰 과적합을 줄입니다.

즉, 이 파라미터들은 점수를 무리하게 train에 맞추기보다 validation AUC가 좋은 지점에서 멈추도록 설계되어 있습니다.


In [12]:
# 모델 학습 함수

def train_catboost_cv(name, variant, seed):
    if CatBoostClassifier is None:
        raise RuntimeError("CatBoost가 설치되어 있지 않습니다.")

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    fold_rows = []
    cat_idx = [X.columns.get_loc(c) for c in cat_cols]

    for fold, (tr, va) in enumerate(cv.split(X, y), start=1):
        model = CatBoostClassifier(**cat_params(seed + fold, variant=variant))

        train_pool = Pool(X.iloc[tr], y[tr], cat_features=cat_idx)
        valid_pool = Pool(X.iloc[va], y[va], cat_features=cat_idx)
        test_pool = Pool(X_test, cat_features=cat_idx)

        try:
            model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
        except Exception as e:
            if USE_CATBOOST_GPU:
                print("GPU 학습 실패. CPU로 재시도합니다:", e)
                params = cat_params(seed + fold, variant=variant)
                params.pop("task_type", None)
                model = CatBoostClassifier(**params)
                model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
            else:
                raise

        pred_va = clip_pred(model.predict_proba(valid_pool)[:, 1])
        pred_te = clip_pred(model.predict_proba(test_pool)[:, 1])

        oof[va] = pred_va
        test_pred += pred_te / N_SPLITS

        auc = roc_auc_score(y[va], pred_va)
        fold_rows.append({"model": name, "seed": seed, "fold": fold, "auc": auc})
        print(f"{name} seed={seed} fold={fold} AUC={auc:.6f}")

    return clip_pred(oof), clip_pred(test_pred), pd.DataFrame(fold_rows)


def train_lgb_cv(name, variant, seed):
    if lgb is None:
        raise RuntimeError("LightGBM이 설치되어 있지 않습니다.")

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    fold_rows = []

    params, early = lgb_params(seed, variant=variant)

    for fold, (tr, va) in enumerate(cv.split(X, y), start=1):
        enc = FoldEncoder(cat_cols, num_cols).fit(X.iloc[tr])
        X_tr = enc.transform(X.iloc[tr])
        X_va = enc.transform(X.iloc[va])
        X_te = enc.transform(X_test)

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr,
            y[tr],
            eval_set=[(X_va, y[va])],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(early, verbose=False)],
        )

        pred_va = clip_pred(model.predict_proba(X_va)[:, 1])
        pred_te = clip_pred(model.predict_proba(X_te)[:, 1])

        oof[va] = pred_va
        test_pred += pred_te / N_SPLITS

        auc = roc_auc_score(y[va], pred_va)
        fold_rows.append({"model": name, "seed": seed, "fold": fold, "auc": auc})
        print(f"{name} seed={seed} fold={fold} AUC={auc:.6f}")

    return clip_pred(oof), clip_pred(test_pred), pd.DataFrame(fold_rows)


def train_xgb_cv(name, variant, seed):
    if XGBClassifier is None:
        raise RuntimeError("XGBoost가 설치되어 있지 않습니다.")

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    fold_rows = []

    for fold, (tr, va) in enumerate(cv.split(X, y), start=1):
        enc = FoldEncoder(cat_cols, num_cols).fit(X.iloc[tr])
        X_tr = enc.transform(X.iloc[tr])
        X_va = enc.transform(X.iloc[va])
        X_te = enc.transform(X_test)

        model = XGBClassifier(**xgb_params(seed + fold, variant=variant))
        model.fit(X_tr, y[tr], eval_set=[(X_va, y[va])], verbose=False)

        pred_va = clip_pred(model.predict_proba(X_va)[:, 1])
        pred_te = clip_pred(model.predict_proba(X_te)[:, 1])

        oof[va] = pred_va
        test_pred += pred_te / N_SPLITS

        auc = roc_auc_score(y[va], pred_va)
        fold_rows.append({"model": name, "seed": seed, "fold": fold, "auc": auc})
        print(f"{name} seed={seed} fold={fold} AUC={auc:.6f}")

    return clip_pred(oof), clip_pred(test_pred), pd.DataFrame(fold_rows)


def train_lgb_gated_cv(name, seed):
    # IVF/DI 시술 유형별로 LightGBM을 따로 학습하는 후보.
    # 한 그룹이 너무 작거나 한 class만 있으면 global fallback을 사용합니다.
    if lgb is None:
        raise RuntimeError("LightGBM이 설치되어 있지 않습니다.")

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    fold_rows = []

    treatment_train = X["시술 유형"].astype(str) if "시술 유형" in X.columns else pd.Series("ALL", index=X.index)
    treatment_test = X_test["시술 유형"].astype(str) if "시술 유형" in X_test.columns else pd.Series("ALL", index=X_test.index)

    params, early = lgb_params(seed, variant="b")

    for fold, (tr, va) in enumerate(cv.split(X, y), start=1):
        pred_va = np.zeros(len(va))
        pred_te = np.zeros(len(X_test))
        te_count = np.zeros(len(X_test))

        # fallback global
        enc_global = FoldEncoder(cat_cols, num_cols).fit(X.iloc[tr])
        X_tr_global = enc_global.transform(X.iloc[tr])
        X_va_global = enc_global.transform(X.iloc[va])
        X_te_global = enc_global.transform(X_test)

        global_model = lgb.LGBMClassifier(**params)
        global_model.fit(
            X_tr_global,
            y[tr],
            eval_set=[(X_va_global, y[va])],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(early, verbose=False)],
        )
        global_va = clip_pred(global_model.predict_proba(X_va_global)[:, 1])
        global_te = clip_pred(global_model.predict_proba(X_te_global)[:, 1])

        pred_va[:] = global_va
        pred_te[:] = global_te
        te_count[:] = 1

        for group_value in ["IVF", "DI"]:
            tr_mask = treatment_train.iloc[tr].values == group_value
            va_mask = treatment_train.iloc[va].values == group_value
            te_mask = treatment_test.values == group_value

            tr_idx_g = np.asarray(tr)[tr_mask]
            va_pos_g = np.where(va_mask)[0]

            if len(tr_idx_g) < 500 or len(np.unique(y[tr_idx_g])) < 2 or len(va_pos_g) == 0:
                continue

            enc = FoldEncoder(cat_cols, num_cols).fit(X.iloc[tr_idx_g])
            X_tr = enc.transform(X.iloc[tr_idx_g])
            X_va = enc.transform(X.iloc[np.asarray(va)[va_pos_g]])
            X_te = enc.transform(X_test.iloc[np.where(te_mask)[0]])

            model = lgb.LGBMClassifier(**params)
            model.fit(X_tr, y[tr_idx_g])

            pred_va[va_pos_g] = clip_pred(model.predict_proba(X_va)[:, 1])
            if X_te.shape[0] > 0:
                pred_te[np.where(te_mask)[0]] = clip_pred(model.predict_proba(X_te)[:, 1])

        oof[va] = pred_va
        test_pred += pred_te / N_SPLITS

        auc = roc_auc_score(y[va], pred_va)
        fold_rows.append({"model": name, "seed": seed, "fold": fold, "auc": auc})
        print(f"{name} seed={seed} fold={fold} AUC={auc:.6f}")

    return clip_pred(oof), clip_pred(test_pred), pd.DataFrame(fold_rows)

## 3. 후보 모델 학습

아래 셀은 시간이 걸립니다.

- `quick`: 구조 확인용
- `full`: 제출 후보 생성용

VS Code에서 `full`을 실행할 경우 시간이 오래 걸릴 수 있으므로,  
필요하면 모델 목록을 줄여서 먼저 실행해도 됩니다.


In [13]:
# 후보 모델 목록

if RUN_MODE == "quick":
    MODEL_PLAN = [
        ("cat_a", "cat", "a"),
        ("xgb_a", "xgb", "a"),
        ("lgb_a", "lgb", "a"),
    ]
else:
    MODEL_PLAN = [
        ("cat_a", "cat", "a"),
        ("cat_b", "cat", "b"),
        ("xgb_a", "xgb", "a"),
        ("xgb_b", "xgb", "b"),
        ("lgb_a", "lgb", "a"),
        ("lgb_b", "lgb", "b"),
        ("lgb_gated", "lgb_gated", "b"),
    ]

print(MODEL_PLAN)

[('cat_a', 'cat', 'a'), ('xgb_a', 'xgb', 'a'), ('lgb_a', 'lgb', 'a')]


In [14]:
# 학습 실행

oof_preds = {}
test_preds = {}
score_rows = []
fold_rows_all = []

for seed in SEEDS:
    for model_name, model_type, variant in MODEL_PLAN:
        run_name = f"{model_name}_seed{seed}"
        print("\n" + "=" * 80)
        print("RUN:", run_name)

        try:
            if model_type == "cat":
                oof, pred_test, fold_df = train_catboost_cv(run_name, variant, seed)
            elif model_type == "xgb":
                oof, pred_test, fold_df = train_xgb_cv(run_name, variant, seed)
            elif model_type == "lgb":
                oof, pred_test, fold_df = train_lgb_cv(run_name, variant, seed)
            elif model_type == "lgb_gated":
                oof, pred_test, fold_df = train_lgb_gated_cv(run_name, seed)
            else:
                raise ValueError(model_type)

            auc = roc_auc_score(y, oof)
            oof_preds[run_name] = oof
            test_preds[run_name] = pred_test

            score_rows.append({
                "run_name": run_name,
                "model_type": model_type,
                "seed": seed,
                "oof_auc": auc,
                "pred_mean": float(np.mean(oof)),
            })
            fold_rows_all.append(fold_df)

            print(f"[DONE] {run_name} OOF AUC={auc:.6f}")

        except Exception as e:
            print(f"[SKIP] {run_name}: {type(e).__name__}: {e}")

score_df = pd.DataFrame(score_rows).sort_values("oof_auc", ascending=False).reset_index(drop=True)
display(score_df)

OUT_DIR.mkdir(parents=True, exist_ok=True)
score_df.to_csv(OUT_DIR / "model_scores.csv", index=False)
if fold_rows_all:
    pd.concat(fold_rows_all, ignore_index=True).to_csv(OUT_DIR / "fold_scores.csv", index=False)


RUN: cat_a_seed42
cat_a_seed42 seed=42 fold=1 AUC=0.738165
cat_a_seed42 seed=42 fold=2 AUC=0.743012
cat_a_seed42 seed=42 fold=3 AUC=0.740060
cat_a_seed42 seed=42 fold=4 AUC=0.738464
cat_a_seed42 seed=42 fold=5 AUC=0.741184
[DONE] cat_a_seed42 OOF AUC=0.740164

RUN: xgb_a_seed42
xgb_a_seed42 seed=42 fold=1 AUC=0.738566
xgb_a_seed42 seed=42 fold=2 AUC=0.742907
xgb_a_seed42 seed=42 fold=3 AUC=0.739965
xgb_a_seed42 seed=42 fold=4 AUC=0.738237
xgb_a_seed42 seed=42 fold=5 AUC=0.741675
[DONE] xgb_a_seed42 OOF AUC=0.740260

RUN: lgb_a_seed42
lgb_a_seed42 seed=42 fold=1 AUC=0.737760
lgb_a_seed42 seed=42 fold=2 AUC=0.742126
lgb_a_seed42 seed=42 fold=3 AUC=0.739918
lgb_a_seed42 seed=42 fold=4 AUC=0.737960
lgb_a_seed42 seed=42 fold=5 AUC=0.740667
[DONE] lgb_a_seed42 OOF AUC=0.739663


,run_name,model_type,seed,oof_auc,pred_mean
0,xgb_a_seed42,xgb,42,0.740260,0.258356
1,cat_a_seed42,cat,42,0.740164,0.258164
2,lgb_a_seed42,lgb,42,0.739663,0.258347


### 5-Fold OOF 학습 결과 해석 방법

학습이 끝나면 `score_df`에 모델별 OOF ROC-AUC가 정리됩니다.

- `fold AUC`: 각 fold validation에서의 ROC-AUC입니다.
- `OOF AUC`: 전체 train 행이 한 번씩 validation 역할을 했을 때의 통합 ROC-AUC입니다.
- `pred_mean`: OOF 예측 확률의 평균으로, 모델이 전체적으로 얼마나 높게 예측하는지 확인하는 보조 지표입니다.

단일 fold 점수보다 OOF AUC가 더 중요합니다.  
OOF AUC는 전체 학습 데이터를 검증에 한 번씩 사용하기 때문에, 단일 train/valid split보다 리더보드 성능을 더 안정적으로 추정할 수 있습니다.


## 4. OOF ROC-AUC 기준 앙상블

개별 모델을 만든 뒤, OOF 예측값을 기준으로 가중 평균 조합을 탐색합니다.

중요한 점은 테스트 점수가 아니라 **학습 데이터의 OOF ROC-AUC**를 기준으로 조합을 선택한다는 것입니다.  
이렇게 하면 제출 파일을 만들기 전, 학습 데이터 안에서 조합의 안정성을 비교할 수 있습니다.


In [15]:
# 앙상블 탐색 함수

def weighted_average(names, weights):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    oof_mat = np.column_stack([oof_preds[n] for n in names])
    test_mat = np.column_stack([test_preds[n] for n in names])
    return clip_pred(oof_mat @ weights), clip_pred(test_mat @ weights)


def softmax_weights(scores, temp=0.00025):
    s = np.asarray(scores, dtype=float)
    z = np.exp((s - s.max()) / temp)
    if not np.all(np.isfinite(z)) or z.sum() <= 0:
        return np.ones_like(s) / len(s)
    return z / z.sum()


def weight_templates(names, auc_lookup):
    k = len(names)
    aucs = np.array([auc_lookup[n] for n in names])
    templates = []

    templates.append(("equal", np.ones(k) / k))
    templates.append(("auc_soft", softmax_weights(aucs, 0.00035)))
    templates.append(("auc_sharp", softmax_weights(aucs, 0.00018)))

    order = np.argsort(-aucs)
    patterns = {
        2: [[0.5, 0.5], [0.6, 0.4], [0.7, 0.3], [0.8, 0.2]],
        3: [[0.5, 0.3, 0.2], [0.45, 0.35, 0.2], [0.6, 0.25, 0.15]],
        4: [[0.4, 0.3, 0.2, 0.1], [0.35, 0.3, 0.2, 0.15], [0.5, 0.25, 0.15, 0.1]],
        5: [[0.32, 0.24, 0.18, 0.14, 0.12], [0.3, 0.25, 0.2, 0.15, 0.1], [0.4, 0.22, 0.16, 0.12, 0.1]],
        6: [[0.28, 0.22, 0.18, 0.14, 0.10, 0.08], [0.35, 0.20, 0.15, 0.12, 0.10, 0.08]],
        7: [[0.24, 0.20, 0.16, 0.13, 0.10, 0.09, 0.08], [0.30, 0.18, 0.15, 0.12, 0.10, 0.08, 0.07]],
    }

    for i, pat in enumerate(patterns.get(k, []), start=1):
        w = np.zeros(k)
        for pos, idx in enumerate(order):
            w[idx] = pat[pos]
        templates.append((f"rank_pattern_{i}", w))

    # 중복 제거
    final = []
    seen = set()
    for label, w in templates:
        w = np.asarray(w, dtype=float)
        w = w / w.sum()
        key = tuple(np.round(w, 5))
        if key not in seen:
            seen.add(key)
            final.append((label, w))
    return final


def blend_search(max_combo_size=6, refine_top=80, random_per_combo=80, seed=42):
    rng = np.random.default_rng(seed)
    names_all = score_df["run_name"].tolist()
    auc_lookup = dict(zip(score_df["run_name"], score_df["oof_auc"]))

    rows = []

    # single
    for n in names_all:
        rows.append({
            "blend_name": f"single_{n}",
            "auc": auc_lookup[n],
            "members": n,
            "weights": "1.0",
        })

    # template search
    for k in range(2, min(max_combo_size, len(names_all)) + 1):
        for combo in itertools.combinations(names_all, k):
            names = list(combo)
            for label, w in weight_templates(names, auc_lookup):
                oof_blend, _ = weighted_average(names, w)
                rows.append({
                    "blend_name": f"k{k}_{label}",
                    "auc": roc_auc_score(y, oof_blend),
                    "members": "|".join(names),
                    "weights": "|".join(f"{x:.8f}" for x in w),
                })

    first = pd.DataFrame(rows).sort_values("auc", ascending=False).reset_index(drop=True)

    # random refinement around high-ranked combinations
    refine_rows = []
    seen_combo = set()

    for _, row in first.head(refine_top).iterrows():
        names = row["members"].split("|")
        if len(names) <= 1:
            continue
        key = tuple(names)
        if key in seen_combo:
            continue
        seen_combo.add(key)

        try:
            base_w = np.array([float(x) for x in row["weights"].split("|")])
            if len(base_w) != len(names):
                continue
        except Exception:
            continue

        base_w = base_w / base_w.sum()

        for i in range(random_per_combo):
            alpha = np.maximum(base_w, 1e-4) * 90
            w = rng.dirichlet(alpha)
            oof_blend, _ = weighted_average(names, w)
            refine_rows.append({
                "blend_name": f"refine_{i:03d}",
                "auc": roc_auc_score(y, oof_blend),
                "members": "|".join(names),
                "weights": "|".join(f"{x:.8f}" for x in w),
            })

    if refine_rows:
        refined = pd.DataFrame(refine_rows)
        summary = pd.concat([first, refined], ignore_index=True)
    else:
        summary = first

    summary = summary.drop_duplicates(["members", "weights"])
    summary = summary.sort_values("auc", ascending=False).reset_index(drop=True)
    return summary


if len(score_df) < 2:
    raise RuntimeError("앙상블을 하려면 완료된 모델이 2개 이상 필요합니다.")

blend_summary = blend_search(
    max_combo_size=4 if RUN_MODE == "quick" else 7,
    refine_top=30 if RUN_MODE == "quick" else 250,
    random_per_combo=30 if RUN_MODE == "quick" else 180,
    seed=42,
)

display(blend_summary.head(20))
blend_summary.to_csv(OUT_DIR / "blend_scores.csv", index=False)

,blend_name,auc,members,weights
0,refine_024,0.740582,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.40061351|0.38254500|0.21684148
1,refine_013,0.740582,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.40796231|0.37829946|0.21373823
2,refine_004,0.740582,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.41129937|0.37534872|0.21335191
3,refine_009,0.740582,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.41189495|0.37271914|0.21538591
4,refine_018,0.740582,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.38228034|0.40065697|0.21706270
5,refine_019,0.740582,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.42094468|0.36392573|0.21512959
6,refine_007,0.740581,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.42041301|0.34954821|0.23003877
7,refine_002,0.740581,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.43315882|0.36126449|0.20557669
8,refine_012,0.740581,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.42246647|0.34083126|0.23670227
9,k3_rank_pattern_2,0.740580,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.45000000|0.35000000|0.20000000


### 앙상블 탐색 결과 해석 방법

`blend_summary`는 여러 모델 조합과 가중치를 OOF ROC-AUC 기준으로 비교한 표입니다.

- `members`: 앙상블에 포함된 모델 목록입니다.
- `weights`: 각 모델의 예측값에 곱한 가중치입니다.
- `auc`: 해당 조합의 OOF ROC-AUC입니다.

가장 위에 있는 조합이 현재 노트북 기준 최종 제출 후보입니다.  
다만 OOF 차이가 매우 작을 때는 리더보드에서 순서가 바뀔 수 있으므로, 제출 횟수를 고려해 상위 1~2개 후보만 제출하는 것이 좋습니다.


In [16]:
# 선택된 앙상블로 제출 파일 생성

selected = blend_summary.iloc[0]
selected_names = selected["members"].split("|")
selected_weights = np.array([float(x) for x in selected["weights"].split("|")])

selected_oof, selected_test = weighted_average(selected_names, selected_weights)
selected_auc = roc_auc_score(y, selected_oof)

print("selected members:")
for n, w in zip(selected_names, selected_weights / selected_weights.sum()):
    print(f"  {n:30s} weight={w:.6f}")

print("selected OOF ROC-AUC:", selected_auc)

pred_col = sample_submission.columns[1]
submission = sample_submission.copy()
submission[pred_col] = selected_test

submission_path = OUT_DIR / "selected_submission.csv"
submission.to_csv(submission_path, index=False)

oof_path = OUT_DIR / "selected_oof.csv"
pd.DataFrame({
    ID_COL: train[ID_COL],
    TARGET: y,
    "prediction": selected_oof,
}).to_csv(oof_path, index=False)

print("saved:", submission_path)
print("saved:", oof_path)
display(submission.head())

selected members:
  xgb_a_seed42                   weight=0.400614
  cat_a_seed42                   weight=0.382545
  lgb_a_seed42                   weight=0.216841
selected OOF ROC-AUC: 0.7405821692576849
saved: c:\Users\jkl83\OneDrive\Desktop\난임 환자 대상 임신 성공 여부 예측 AI 프로젝트\outputs\project_fertility_psp_v4\selected_submission.csv
saved: c:\Users\jkl83\OneDrive\Desktop\난임 환자 대상 임신 성공 여부 예측 AI 프로젝트\outputs\project_fertility_psp_v4\selected_oof.csv


,ID,probability
0,TEST_00000,0.001621
1,TEST_00001,0.003741
2,TEST_00002,0.168677
3,TEST_00003,0.106717
4,TEST_00004,0.514805


### 제출 파일 생성 해석

이 셀은 OOF에서 선택된 최적 앙상블 조합을 제출용 예측값에 동일하게 적용합니다.

- `selected_names`: 최종 선택된 모델 목록입니다.
- `selected_weights`: OOF 기준으로 선택된 모델별 가중치입니다.
- `selected_oof`: train 전체에 대한 OOF 예측값입니다.
- `selected_test`: 제출용 데이터에 대한 최종 예측값입니다.

제출 파일은 sample submission의 구조를 그대로 복사한 뒤, 확률 컬럼만 예측값으로 교체합니다.  
따라서 최종 CSV는 반드시 `ID`와 확률 컬럼 두 개만 가져야 합니다.


In [17]:
# 제출 파일 검증

pred_col = sample_submission.columns[1]
print("submission shape:", submission.shape)
print("expected rows:", len(sample_submission))
print("NaN count:", int(submission[pred_col].isna().sum()))
print("min:", float(submission[pred_col].min()))
print("max:", float(submission[pred_col].max()))
print("mean:", float(submission[pred_col].mean()))

assert len(submission) == len(sample_submission)
assert submission[pred_col].isna().sum() == 0
assert submission[pred_col].between(0, 1).all()

print("검증 완료")

submission shape: (90067, 2)
expected rows: 90067
NaN count: 0
min: 0.0006425714859835343
max: 0.7520534210941635
mean: 0.25825811153175937
검증 완료


## 추가: Rank Average 앙상블 추가

v4는 rankdata를 import해놓고 실제 앙상블에는 주로 확률 가중평균을 씁니다.
ROC-AUC는 순위 기반이라, Rank Average가 LB에서 더 잘 나올 수 있어서 써보겠습니다. (정)

In [19]:
# Rank Average 앙상블 탐색

def rank_average(names, weights):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()

    oof_rank_mat = np.column_stack([
        rankdata(oof_preds[n]) / len(oof_preds[n])
        for n in names
    ])

    test_rank_mat = np.column_stack([
        rankdata(test_preds[n]) / len(test_preds[n])
        for n in names
    ])

    return clip_pred(oof_rank_mat @ weights), clip_pred(test_rank_mat @ weights)


rank_rows = []

for _, row in blend_summary.head(300).iterrows():
    names = row["members"].split("|")
    weights = np.array([float(x) for x in row["weights"].split("|")])

    rank_oof, rank_test = rank_average(names, weights)

    rank_rows.append({
        "blend_name": "rank_" + row["blend_name"],
        "auc": roc_auc_score(y, rank_oof),
        "members": row["members"],
        "weights": row["weights"],
    })

rank_blend_summary = (
    pd.DataFrame(rank_rows)
    .sort_values("auc", ascending=False)
    .reset_index(drop=True)
)

display(rank_blend_summary.head(20))
rank_blend_summary.to_csv(OUT_DIR / "rank_blend_scores.csv", index=False)

,blend_name,auc,members,weights
0,rank_refine_024,0.740593,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.40061351|0.38254500|0.21684148
1,rank_refine_009,0.740593,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.41189495|0.37271914|0.21538591
2,rank_refine_013,0.740593,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.40796231|0.37829946|0.21373823
3,rank_refine_004,0.740593,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.41129937|0.37534872|0.21335191
4,rank_refine_019,0.740593,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.42094468|0.36392573|0.21512959
5,rank_refine_018,0.740593,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.38228034|0.40065697|0.21706270
6,rank_refine_007,0.740592,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.42041301|0.34954821|0.23003877
7,rank_refine_002,0.740592,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.43315882|0.36126449|0.20557669
8,rank_refine_012,0.740591,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.42246647|0.34083126|0.23670227
9,rank_k3_rank_pattern_2,0.740591,xgb_a_seed42|cat_a_seed42|lgb_a_seed42,0.45000000|0.35000000|0.20000000


### 제출 전 검증 포인트

제출 전에는 아래 조건을 반드시 확인합니다.

- sample submission과 행 수가 같은가?
- 확률 컬럼에 결측값이 없는가?
- 예측값이 0 이상 1 이하인가?
- ID 순서가 sample submission과 동일한가?

이 조건이 깨지면 모델 성능과 무관하게 제출 파일 형식 오류가 발생할 수 있습니다.


In [20]:
# Rank ensemble 최상위 조합으로 제출 파일 생성

selected_rank = rank_blend_summary.iloc[0]

selected_names = selected_rank["members"].split("|")
selected_weights = np.array([
    float(x) for x in selected_rank["weights"].split("|")
])

selected_oof_rank, selected_test_rank = rank_average(
    selected_names,
    selected_weights
)

print("selected rank blend:", selected_rank["blend_name"])
print("OOF ROC-AUC:", roc_auc_score(y, selected_oof_rank))

for name, weight in zip(selected_names, selected_weights / selected_weights.sum()):
    print(f"{name}: {weight:.6f}")

pred_col = sample_submission.columns[1]

submission_rank = sample_submission.copy()
submission_rank[pred_col] = selected_test_rank

rank_submission_path = OUT_DIR / "submission_rank_refine_024.csv"
submission_rank.to_csv(rank_submission_path, index=False)

print("저장 완료:", rank_submission_path)
display(submission_rank.head())

selected rank blend: rank_refine_024
OOF ROC-AUC: 0.7405932740805476
xgb_a_seed42: 0.400614
cat_a_seed42: 0.382545
lgb_a_seed42: 0.216841
저장 완료: c:\Users\jkl83\OneDrive\Desktop\난임 환자 대상 임신 성공 여부 예측 AI 프로젝트\outputs\project_fertility_psp_v4\submission_rank_refine_024.csv


,ID,probability
0,TEST_00000,0.075806
1,TEST_00001,0.114697
2,TEST_00002,0.292184
3,TEST_00003,0.206363
4,TEST_00004,0.957397


In [21]:
# 제출 파일 검증

submission_check = pd.read_csv(rank_submission_path)

print("shape:", submission_check.shape)
print("columns:", submission_check.columns.tolist())
print("ID 일치:", (submission_check[ID_COL] == sample_submission[ID_COL]).all())
print("결측값 수:", submission_check[pred_col].isna().sum())
print("확률 범위:", submission_check[pred_col].min(), "~", submission_check[pred_col].max())

display(submission_check[pred_col].describe())
display(submission_check.head())

shape: (90067, 2)
columns: ['ID', 'probability']
ID 일치: True
결측값 수: 0
확률 범위: 0.0026408724880206 ~ 0.9999927773274728


count    90067.000000
mean         0.500006
std          0.288036
min          0.002641
25%          0.251380
50%          0.500500
75%          0.749458
max          0.999993
Name: probability, dtype: float64

,ID,probability
0,TEST_00000,0.075806
1,TEST_00001,0.114697
2,TEST_00002,0.292184
3,TEST_00003,0.206363
4,TEST_00004,0.957397


## 정리

이 노트북은 처음 받은 데이터에서부터 다음 과정을 순서대로 재현합니다.

1. **EDA 기반 feature 처리**
- 시술 유형, 나이, 결측 패턴, 배아/난자 변수의 성공률 차이를 feature 설계에 반영합니다.

2. **시술 구조와 결측 신호 활용**
- DI/IVF 구조 차이, 배아 관련 구조적 결측, 고결측 유전검사 변수의 known/missing 여부를 모델 입력으로 사용합니다.

3. **범주형 변수 처리**
- CatBoost는 문자열 범주형을 직접 사용합니다.
- LightGBM/XGBoost는 fold 내부 train 기준 ordinal/frequency encoding을 적용합니다.

4. **5-Fold OOF 검증**
- 단일 train/valid split이 아니라 전체 train 데이터를 한 번씩 validation으로 사용하는 OOF ROC-AUC를 핵심 기준으로 삼습니다.

5. **OOF 기준 앙상블**
- CatBoost, LightGBM, XGBoost 및 seed/variant별 후보 모델을 조합하여 OOF ROC-AUC가 가장 높은 weighted ensemble을 선택합니다.

6. **제출 파일 생성**
- OOF에서 선택한 동일한 모델 조합과 가중치를 제출용 예측값에 적용해 최종 CSV를 생성합니다.

최종적으로 이 노트북의 핵심 전략은 **임상적 의미가 있는 feature engineering + 누수 없는 OOF 검증 + 모델 다양성을 활용한 앙상블**입니다.
